<a href="https://colab.research.google.com/github/AnishPanicker/cancer-classifier-api/blob/main/API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi pyngrok nest-asyncio pillow numpy -q

In [ ]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, File, UploadFile
import numpy as np
from io import BytesIO
from PIL import Image
import keras
import uvicorn
from pyngrok import ngrok

In [ ]:
MODEL = keras.models.load_model("/content/drive/MyDrive/CNN/lung_and_colon_cancer-main/training/model/model.keras")
CLASS_NAMES = ["Colon Adenocarcinoma", "Colon Benign Tissue", "Lung Adenocarcinoma", "Lung Benign Tissue", "Lung Squamous Cell Carcinoma"]
IMG_SIZE = 64

In [ ]:
app = FastAPI()

def read_file_as_image(data) -> np.ndarray:
    image = Image.open(BytesIO(data))
    image = image.resize((IMG_SIZE, IMG_SIZE))
    return np.array(image)

@app.get("/ping")
async def ping():
    return "Server is alive"

@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    image = read_file_as_image(await file.read())
    img_batch = np.expand_dims(image, axis=0)
    predictions = MODEL.predict(img_batch)
    predicted_class = CLASS_NAMES[np.argmax(predictions[0])]
    confidence = float(np.max(predictions[0]))
    return {"class": predicted_class, "confidence": round(confidence * 100, 2)}

In [ ]:
#Use your token here I hid it.
#ngrok.set_auth_token("################################################")
public_url = ngrok.connect(8000)
print("Your API URL:", public_url)

Your API URL: NgrokTunnel: "https://noneffusive-nonevolving-toccara.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
async def start():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    await server.serve()

asyncio.get_event_loop().run_until_complete(start())

INFO:     Started server process [334]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /ping HTTP/1.1" 200 OK
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /openapi.json HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "POST /predict HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /openapi.json HTTP/1.1" 200 OK
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     2401:9640:0:ca3b:2:2:2:1:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     146.196.39.49:0 - "GE